# 04 – RFM Feature Engineering



## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Locate project and load cleaned transactions

In [2]:
current_path = Path.cwd()
project_root = None

for candidate in [current_path, *current_path.parents]:
    if (candidate / "data" / "interim").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find data/interim. Open this notebook from inside your project folder."
    )

input_file = project_root / "data" / "interim" / "cleaned_transactions.csv"

if not input_file.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {input_file}")

df = pd.read_csv(input_file, low_memory=False)

print("Loaded:", input_file)
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())

Loaded: d:\loyality_reward_optimization\data\interim\cleaned_transactions.csv
Rows: 1033036
Columns: 16


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,source_sheet,is_cancelled,transaction_type,line_amount,is_zero_price,is_negative_price,is_customer_id_missing,is_rfm_eligible
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,83.40,False,False,False,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,81.00,False,False,False,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,81.00,False,False,False,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,100.80,False,False,False,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,30.00,False,False,False,True


## 3. Identify and validate required columns

In [3]:
df.columns = [str(col).strip().lower().replace(" ", "_") for col in df.columns]

def find_column(possible_names):
    for name in possible_names:
        if name in df.columns:
            return name
    return None

customer_col = find_column(["customer_id", "customerid", "customer"])
invoice_col = find_column(["invoice", "invoiceno", "invoice_no"])
date_col = find_column(["invoice_date", "invoicedate", "date"])
quantity_col = find_column(["quantity", "qty"])
price_col = find_column(["price", "unit_price"])
amount_col = find_column(["line_amount", "total_amount", "amount"])
type_col = find_column(["transaction_type"])
eligible_col = find_column(["is_rfm_eligible"])

required_columns = {
    "customer_id": customer_col,
    "invoice": invoice_col,
    "invoice_date": date_col,
    "quantity": quantity_col,
    "price": price_col
}

missing_columns = [name for name, col in required_columns.items() if col is None]

if missing_columns:
    raise ValueError("Missing required columns: " + ", ".join(missing_columns))

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df[customer_col] = pd.to_numeric(df[customer_col], errors="coerce")
df[quantity_col] = pd.to_numeric(df[quantity_col], errors="coerce")
df[price_col] = pd.to_numeric(df[price_col], errors="coerce")

if amount_col is None:
    amount_col = "line_amount"
    df[amount_col] = df[quantity_col] * df[price_col]
else:
    df[amount_col] = pd.to_numeric(df[amount_col], errors="coerce")

print("Required columns validated.")
print("Customer column:", customer_col)
print("Invoice column:", invoice_col)
print("Date column:", date_col)
print("Amount column:", amount_col)
print("Eligibility flag found:", eligible_col is not None)

Required columns validated.
Customer column: customer_id
Invoice column: invoice
Date column: invoicedate
Amount column: line_amount
Eligibility flag found: True


## 4. Select qualifying purchase transactions

In [4]:
if eligible_col:
    eligibility = df[eligible_col].astype(str).str.strip().str.lower().isin(
        ["true", "1", "yes"]
    )
    qualifying = df[eligibility].copy()
else:
    qualifying = df[
        df[customer_col].notna()
        & df[date_col].notna()
        & (df[quantity_col] > 0)
        & (df[price_col] > 0)
    ].copy()

    if type_col:
        qualifying = qualifying[
            qualifying[type_col].astype(str).str.lower() == "normal_sale"
        ].copy()

# Recalculate line amount consistently from quantity × unit price.
qualifying[amount_col] = qualifying[quantity_col] * qualifying[price_col]

# Final defensive checks.
qualifying = qualifying[
    qualifying[customer_col].notna()
    & qualifying[date_col].notna()
    & qualifying[invoice_col].notna()
    & (qualifying[quantity_col] > 0)
    & (qualifying[price_col] > 0)
    & (qualifying[amount_col] > 0)
].copy()

print("Total cleaned rows:", len(df))
print("Qualifying purchase rows:", len(qualifying))
print("Customers included:", qualifying[customer_col].nunique())
print("Invoices included:", qualifying[invoice_col].nunique())

if qualifying.empty:
    raise ValueError("No qualifying purchase rows found. Check the eligibility logic.")

display(qualifying.head())

Total cleaned rows: 1033036
Qualifying purchase rows: 779425
Customers included: 5878
Invoices included: 36969


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,source_sheet,is_cancelled,transaction_type,line_amount,is_zero_price,is_negative_price,is_customer_id_missing,is_rfm_eligible
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,83.40,False,False,False,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,81.00,False,False,False,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,81.00,False,False,False,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,100.80,False,False,False,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,Year 2009-2010,False,normal_sale,30.00,False,False,False,True


## 5. Choose the RFM reference date

In [5]:
latest_purchase_date = qualifying[date_col].max()
reference_date = latest_purchase_date.normalize() + pd.Timedelta(days=1)

print("Latest qualifying purchase:", latest_purchase_date)
print("RFM reference date:", reference_date)

Latest qualifying purchase: 2011-12-09 12:50:00
RFM reference date: 2011-12-10 00:00:00


## 6. Calculate customer-level RFM

In [6]:
rfm = (
    qualifying.groupby(customer_col)
    .agg(
        last_purchase_date=(date_col, "max"),
        frequency=(invoice_col, "nunique"),
        monetary=(amount_col, "sum")
    )
    .reset_index()
)

rfm["recency"] = (
    reference_date - rfm["last_purchase_date"].dt.normalize()
).dt.days

# Keep a clear, consistent customer identifier column.
rfm = rfm.rename(columns={customer_col: "customer_id"})

rfm = rfm[
    ["customer_id", "recency", "frequency", "monetary", "last_purchase_date"]
].sort_values("customer_id").reset_index(drop=True)

display(rfm.head(10))
print("RFM rows:", len(rfm))
print("Unique RFM customers:", rfm["customer_id"].nunique())

,customer_id,recency,frequency,monetary,last_purchase_date
0,"12,346.00",326,12,"77,556.46",2011-01-18 10:01:00
1,"12,347.00",3,8,"4,921.53",2011-12-07 15:52:00
2,"12,348.00",76,5,"2,019.40",2011-09-25 13:13:00
3,"12,349.00",19,4,"4,428.69",2011-11-21 09:51:00
4,"12,350.00",311,1,334.40,2011-02-02 16:01:00
5,"12,351.00",376,1,300.93,2010-11-29 15:23:00
6,"12,352.00",37,10,"2,849.84",2011-11-03 14:37:00
7,"12,353.00",205,2,406.76,2011-05-19 17:47:00
8,"12,354.00",233,1,"1,079.40",2011-04-21 13:11:00
9,"12,355.00",215,2,947.61,2011-05-09 13:49:00


RFM rows: 5878
Unique RFM customers: 5878


## 7. Validate RFM values

In [7]:
validation = {
    "No missing customer IDs": rfm["customer_id"].notna().all(),
    "No missing recency": rfm["recency"].notna().all(),
    "No negative recency": (rfm["recency"] >= 0).all(),
    "Frequency is positive": (rfm["frequency"] > 0).all(),
    "Monetary is positive": (rfm["monetary"] > 0).all(),
    "One row per customer": rfm["customer_id"].is_unique
}

for check, passed in validation.items():
    print(("PASS" if passed else "FAIL") + " - " + check)

if not all(validation.values()):
    raise ValueError("RFM validation failed. Review the checks above.")

print("\nAll RFM validation checks passed.")

PASS - No missing customer IDs
PASS - No missing recency
PASS - No negative recency
PASS - Frequency is positive
PASS - Monetary is positive
PASS - One row per customer

All RFM validation checks passed.


## 8. RFM descriptive statistics

In [8]:
rfm_summary = rfm[["recency", "frequency", "monetary"]].describe().T
display(rfm_summary)

rfm_summary.to_csv(
    project_root / "data" / "processed" / "rfm_feature_summary.csv"
)

,count,mean,std,min,25%,50%,75%,max
recency,"5,878.00",201.87,209.35,1.00,26.00,96.00,380.00,739.00
frequency,"5,878.00",6.29,13.01,1.00,1.00,3.00,7.00,398.00
monetary,"5,878.00","2,955.90","14,440.85",2.95,342.28,867.74,"2,248.30","580,987.04"


## 9. Save the RFM customer dataset as CSV

In [9]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "rfm_customer_features.csv"
rfm.to_csv(output_file, index=False)

# Reload the CSV and verify the saved file.
rfm_check = pd.read_csv(output_file)

print("Saved file:", output_file)
print("Saved rows:", len(rfm_check))
print("Saved columns:", rfm_check.columns.tolist())
print("Unique saved customers:", rfm_check["customer_id"].nunique())

assert len(rfm_check) == len(rfm)
assert rfm_check["customer_id"].is_unique

print("CSV save and reload validation: PASSED")
display(rfm_check.head())

Saved file: d:\loyality_reward_optimization\data\processed\rfm_customer_features.csv
Saved rows: 5878
Saved columns: ['customer_id', 'recency', 'frequency', 'monetary', 'last_purchase_date']
Unique saved customers: 5878
CSV save and reload validation: PASSED


,customer_id,recency,frequency,monetary,last_purchase_date
0,"12,346.00",326,12,"77,556.46",2011-01-18 10:01:00
1,"12,347.00",3,8,"4,921.53",2011-12-07 15:52:00
2,"12,348.00",76,5,"2,019.40",2011-09-25 13:13:00
3,"12,349.00",19,4,"4,428.69",2011-11-21 09:51:00
4,"12,350.00",311,1,334.40,2011-02-02 16:01:00
